# Defaqto Sales DB — 02 · The same transforms as a dbt project

**Author:** Ketki Kothe · Solution Engineer, Snowflake · `Snowflake Solution Engineering`  
**Workshop:** Snowflake × Defaqto, Tuesday 8 September 2026, Snowflake London  
**Aggregation logic:** supplied by the Defaqto data lead

In notebook 01 you built the aggregate with Dynamic Tables. This notebook builds the
**same output** as a dbt project running natively inside Snowflake — no dbt installed
anywhere, no orchestration server.

The reason to look at both: they are good at different things, and the difference is
not what people expect.

| | Dynamic Tables (01) | dbt project (02) |
| --- | --- | --- |
| Refresh | automatic, `TARGET_LAG` | on demand, or scheduled by a Task |
| Tests | none — they compute, they don't assert | **16 tests, built in** |
| Lineage | Snowsight object graph | dbt DAG, plus the object graph |
| Version control | the DDL is the definition | git, with review |

**The line that matters:** a dynamic table cannot tell you your data is wrong. This
project ships a test that finds roughly 10,000 rows carrying about £1.6m of premium
that does not belong where a naive join would put it.

### Before you run anything

**Edit the next cell** — put your own alias in it, same as notebook 01.

In [ ]:
# ===========================================================================
# THE ONLY CELL YOU NEED TO EDIT.
# Replace "kkothe" with your own alias - the same one you used in notebook 01.
# ===========================================================================
alias = "kkothe"

# Everything below is derived from it. Leave it alone.
#
# This is a Python cell rather than SQL because EXECUTE DBT PROJECT requires ARGS
# to be a string literal - it rejects `'run --vars {alias:' || $alias || '}'` with
# a syntax error. A Python variable rendered through Jinja is substituted before
# Snowflake parses the statement, so the alias still lives in one place.
staging_schema = f"DBT_{alias.upper()}_STAGING"
marts_schema   = f"DBT_{alias.upper()}_MARTS"

print(f"alias           : {alias}")
print(f"staging schema  : DEFAQTO_DB.{staging_schema}")
print(f"marts schema    : DEFAQTO_DB.{marts_schema}")
print(f"dynamic tables  : DEFAQTO_DB.TRANSFORMED_{alias.upper()}   (from notebook 01)")

## Step 1 — What is in the project

Nine files, already deployed to Snowflake as `DEFAQTO_DB.DBT.DEFAQTO_SALESDB_DBT`.

```
dbt/defaqto_salesdb/
├── dbt_project.yml                  project config, schema routing
├── profiles.yml                     no password, no authenticator, no env_var()
├── macros/
│   └── generate_schema_name.sql     routes your models to YOUR schema
├── models/
│   ├── schema.yml                   sources, 16 tests, column docs
│   ├── staging/
│   │   └── stg_sales_events.sql     sale_date derived - a VIEW, not a table
│   └── marts/
│       ├── salesdb_aggregate.sql    Mike's rollup
│       └── fct_stc_funnel_daily.sql quotes -> priced -> clicked -> sold
└── tests/
    ├── assert_distinct_quotes_within_row_count.sql      expected to PASS
    └── assert_no_cross_product_quote_collision.sql      expected to WARN
```

Two things worth noticing before you run it.

**`profiles.yml` has no credentials.** For Snowflake-native dbt there is no
`password`, no `authenticator` and no `env_var()` — dbt runs inside Snowflake,
authenticated by the session that invokes it. Any of those three fails the deploy.

**The staging model is a view, not a table.** In notebook 01 the staging layer *had*
to be materialised, purely so incremental refresh could group on a simple column. dbt
rebuilds the mart on each run, so that workaround is unnecessary — the view costs no
storage. Same logic, one fewer compromise.

## Step 2 — Run it

`EXECUTE DBT PROJECT` runs the deployed project. Nothing is installed locally.

**One piece of syntax to get right.** dbt vars must have **no space after the colon**:

| | |
| --- | --- |
| `ARGS='run --vars {alias: kkothe}'` | fails — `String '{alias:' is not valid YAML` |
| `ARGS='run --vars {alias:kkothe}'` | works |

---

### Option: run it from Cortex Code instead

```text
In Snowflake account <your-account>, run the deployed dbt project
DEFAQTO_DB.DBT.DEFAQTO_SALESDB_DBT twice: first `run`, then `test`, passing my alias
as a dbt var so my models land in my own schema.

Use EXECUTE DBT PROJECT with ARGS as a single-quoted string literal, and note that
the var must have NO space after the colon - {alias:myalias} works, {alias: myalias}
fails as invalid YAML.

Then show me which tests passed and which warned, and explain what the warning found.
Do not modify anything in DEFAQTO_DB.RAW.
```

In [ ]:
%%sql -r dataframe_1
-- Builds three models: one view in your STAGING schema, two tables in your MARTS
-- schema. Expect PASS=3 WARN=0 ERROR=0.
--
-- {{alias}} is substituted by the notebook before Snowflake sees this, because ARGS
-- must be a literal string.
EXECUTE DBT PROJECT DEFAQTO_DB.DBT.DEFAQTO_SALESDB_DBT
  ARGS='run --vars {alias:{{alias}}}';

In [ ]:
%%sql -r dataframe_2
-- Runs all 16 tests. Expect PASS=15 WARN=1 ERROR=0.
--
-- The single warning is deliberate and is the most interesting output in this
-- notebook - see the next cell.
EXECUTE DBT PROJECT DEFAQTO_DB.DBT.DEFAQTO_SALESDB_DBT
  ARGS='test --vars {alias:{{alias}}}';

## Step 3 — What the warning found

`PASS=15 WARN=1 ERROR=0`. Fifteen tests pass. One warns, and that one is the point of
the whole notebook.

`assert_no_cross_product_quote_collision` is configured `severity: warn` rather than
`error`, on purpose. What it finds is a fact about the source data, not a defect this
project introduced — so it should be surfaced on every single run without failing the
build. Run the next cell to see what it caught.

In [ ]:
%%sql -r dataframe_3
-- QUOTE_ID is minted per product database, so a Gap quote and a short-term car quote
-- can carry the same number while referring to entirely different things.
--
-- Joining sales to STC_QUOTES on quote_id alone therefore produces matches that are
-- pure coincidence. These are those matches.
SELECT s.PRODUCT_TYPE_NAME,
       COUNT(*)                       AS colliding_rows,
       ROUND(SUM(s.GWP))              AS gwp_wrongly_attributed
FROM DEFAQTO_DB.{{marts_schema | replace('MARTS','STAGING')}}.STG_SALES_EVENTS s
JOIN DEFAQTO_DB.RAW.STC_QUOTES q ON q.QUOTE_ID = s.QUOTE_ID
WHERE s.PRODUCTTYPE_ID <> 11
GROUP BY 1
ORDER BY colliding_rows DESC;

In [ ]:
%%sql -r dataframe_4
-- Does dbt produce the same answer as the Dynamic Tables you built in notebook 01?
-- Both must also match Mike's original SQL. All three difference counts must be 0.
WITH mike AS (
    SELECT CAST(date_time AS DATE) AS sale_date, affiliate_id, affiliate_name,
           provider_id, provider_name, producttype_id AS product_type_id,
           product_type_name, cancellation,
           COUNT(*) AS row_count, COUNT(DISTINCT quote_id) AS distinct_quotes,
           SUM(gwp) AS total_gwp, SUM(commission) AS total_commission
    FROM DEFAQTO_DB.RAW.SALESDB_SALESEVENTS GROUP BY 1,2,3,4,5,6,7,8
),
dbt AS (SELECT * FROM DEFAQTO_DB.{{marts_schema}}.SALESDB_AGGREGATE),
-- Notebook 01 renamed this to SILVER_SALESDB_AGGREGATE when the aggregate moved
-- into the silver layer. Column ORDER still matches, which is what MINUS compares.
dt  AS (SELECT * FROM DEFAQTO_DB.TRANSFORMED_{{alias | upper}}.SILVER_SALESDB_AGGREGATE)
SELECT 'dbt rows'                       AS check_name, COUNT(*)::VARCHAR AS value FROM dbt
UNION ALL SELECT 'dynamic table rows',   COUNT(*)::VARCHAR FROM dt
UNION ALL SELECT 'mike SQL rows',        COUNT(*)::VARCHAR FROM mike
UNION ALL SELECT 'dbt minus dynamic table',  (SELECT COUNT(*) FROM (SELECT * FROM dbt MINUS SELECT * FROM dt))::VARCHAR
UNION ALL SELECT 'dynamic table minus dbt',  (SELECT COUNT(*) FROM (SELECT * FROM dt MINUS SELECT * FROM dbt))::VARCHAR
UNION ALL SELECT 'dbt minus mike SQL',       (SELECT COUNT(*) FROM (SELECT * FROM dbt MINUS SELECT * FROM mike))::VARCHAR;

## Step 4 — See the DAG and the docs in Snowsight

### The dbt project object

1. Sign in to **Snowsight**
2. **Catalog » Database Explorer**
3. `DEFAQTO_DB` » `DBT` — the project appears as **`DEFAQTO_SALESDB_DBT`**, a first-class
   Snowflake object with versions, not a folder on someone's laptop
4. Each `snow dbt deploy` adds a version — `SHOW VERSIONS IN DBT PROJECT` lists them

### The dbt DAG

1. **Projects » Workspaces**
2. Open the workspace containing the dbt files
3. Select the project and profile, then **Compile**
4. Select the **DAG** tab below the editor

You get sources → staging → marts, derived from the `ref()` and `source()` calls in
the models. Nobody drew it.

### Your models

1. **Catalog » Database Explorer**
2. `DEFAQTO_DB` » your **`DBT_<alias>_MARTS`** schema
3. `SALESDB_AGGREGATE` and `FCT_STC_FUNNEL_DAILY` are ordinary tables — the column
   descriptions from `schema.yml` carry through as comments

## Step 5 — Scheduling, if this were production

Not run here — six attendees each creating a task would leave six schedules running
after the workshop. This is what it would look like:

```sql
CREATE OR REPLACE TASK DEFAQTO_DB.DBT.RUN_SALESDB_DBT_DAILY
  WAREHOUSE = COMPUTE_WH
  SCHEDULE  = 'USING CRON 0 6 * * * UTC'      -- 06:00 UTC daily
AS
EXECUTE DBT PROJECT DEFAQTO_DB.DBT.DEFAQTO_SALESDB_DBT
  ARGS='build --vars {alias:prod}';

ALTER TASK DEFAQTO_DB.DBT.RUN_SALESDB_DBT_DAILY RESUME;
```

`build` rather than `run`, because `build` runs the models **and the tests** in
dependency order. A scheduled `run` on its own would rebuild the marts every morning
and never check whether the data was sane.

Monitor it under **Transformation » Tasks**, which gives run history, duration trends
and retry — the observability the current cron job does not have.

## Done — and which should Defaqto actually use?

Both, for different jobs. The honest split:

**Dynamic Tables** when freshness is the requirement and the logic is a
transformation. Declare a `TARGET_LAG` and stop thinking about scheduling. That is
the right answer for anything feeding a live dashboard.

**dbt** when correctness needs to be *asserted*, not assumed, and when changes need
review before they land. The 16 tests in this project are the difference: a dynamic
table computes, it does not tell you the answer is wrong.

They are not mutually exclusive — dbt can materialise a model **as** a dynamic table,
which gives you tested definitions in version control and automatic refresh.

### The question that decides notebook 01's design

The dynamic tables are `INCREMENTAL`, which suits an append-only feed. If the
production load **truncates and reloads** each night, every row changes every night,
incremental becomes slower than full, and the answer is a single full-refresh table
instead of two. **That is a question for Mike**, and it is the one input that settles it.

### Housekeeping

Your objects are in `DBT_{alias}_STAGING`, `DBT_{alias}_MARTS` and
`TRANSFORMED_{alias}`. To remove them after the day:

```sql
DROP SCHEMA IF EXISTS DEFAQTO_DB.DBT_<alias>_STAGING;
DROP SCHEMA IF EXISTS DEFAQTO_DB.DBT_<alias>_MARTS;
DROP SCHEMA IF EXISTS DEFAQTO_DB.TRANSFORMED_<alias>;
```

`DEFAQTO_DB.RAW` stays. It is shared, and nothing in either notebook writes to it.

---

*Built by Ketki Kothe · Solution Engineer, Snowflake · `Snowflake Solution Engineering`*